# ⚡ Análise de Consumo de Energia Elétrica no Brasil
## Projeto G2 — Tema 14

---

### 1. Introdução

Este notebook realiza uma análise exploratória completa do consumo de energia elétrica no Brasil entre **2015 e 2024**,
respondendo às principais perguntas do projeto:

- Quais estados consomem mais energia?
- Quais setores apresentam maior consumo?
- Existe sazonalidade no consumo?
- O consumo aumentou ao longo do tempo?
- Há relação entre temperatura e consumo?
- Existem períodos de pico de demanda?
- Quais regiões apresentam maior crescimento energético?

### 2. Contextualização Energética

O Brasil possui uma matriz elétrica diversificada, com predominância de fontes renováveis (hidrelétricas ~60%, eólica e solar em expansão).
O consumo de energia é um indicador econômico e social relevante, refletindo:

- **Crescimento urbano** e industrial
- **Desenvolvimento econômico** regional
- **Comportamento de consumo** das famílias
- **Eficiência energética** de processos produtivos

Monitorar o consumo permite identificar padrões de demanda, sazonalidade e riscos de sobrecarga no sistema elétrico.

### 3. Explicação da Base de Dados

O dataset `simulacao_consumo_energia_brasil.csv` contém registros mensais por estado e setor econômico, com as colunas:

| Coluna | Tipo | Descrição |
|---|---|---|
| ano / mes / data | int / datetime | Período de referência |
| regiao / uf | str | Localização geográfica |
| setor_consumo | str | Residencial, Industrial, Comercial, Rural, Público |
| consumo_mwh | float | Consumo em MWh |
| demanda_pico | float | Demanda máxima registrada |
| temperatura_media | float | Temperatura média (°C) |
| tarifa_media | float | Tarifa média (R$/kWh) |
| populacao | int | População estimada |
| eficiencia_energetica | float | Índice 0–100 |
| emissao_co2 | float | Emissão estimada de CO₂ |
| nivel_demanda | str | Baixo / Médio / Alto / Crítico |

### 4. Importações e Configuração

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
print('✅ Bibliotecas importadas com sucesso!')

### 5. Leitura dos Dados

In [ ]:
df = pd.read_csv('../dados/simulacao_consumo_energia_brasil.csv', parse_dates=['data'])

print(f'📦 Shape do dataset: {df.shape}')
print(f'📅 Período: {df["ano"].min()} a {df["ano"].max()}')
print(f'🗺️  Regiões: {sorted(df["regiao"].unique())}')
print(f'📍 Estados: {sorted(df["uf"].unique())}')
print(f'🏭 Setores: {sorted(df["setor_consumo"].unique())}')
df.head(10)

### 6. Limpeza e Preparação dos Dados

In [ ]:
# Verificar valores nulos
print('🔍 Valores nulos por coluna:')
print(df.isnull().sum())

# Verificar duplicatas
print(f'\n🔁 Duplicatas: {df.duplicated().sum()}')

# Tipos de dados
print('\n📊 Tipos de dados:')
print(df.dtypes)

In [ ]:
# Estatísticas descritivas
df.describe()

### 7. Engenharia de Atributos

In [ ]:
# Consumo per capita
df['consumo_per_capita'] = df['consumo_mwh'] / df['populacao'] * 1000  # kWh per capita

# Nome do mês
meses_pt = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
            7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}
df['mes_nome'] = df['mes'].map(meses_pt)

# Período (ano-mês)
df['periodo'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes'].astype(str) + '-01')

# Trimestre
df['trimestre'] = df['mes'].apply(lambda m: f'Q{(m-1)//3+1}')

print('✅ Novas colunas criadas: consumo_per_capita, mes_nome, periodo, trimestre')
df[['ano','mes','mes_nome','trimestre','consumo_mwh','consumo_per_capita']].head()

### 8. KPIs

In [ ]:
kpis = {
    'Consumo Total (TWh)': df['consumo_mwh'].sum() / 1e6,
    'Estado com Maior Consumo': df.groupby('uf')['consumo_mwh'].sum().idxmax(),
    'Setor Mais Consumidor': df.groupby('setor_consumo')['consumo_mwh'].sum().idxmax(),
    'Demanda Média (MW)': df['demanda_pico'].mean(),
    'Tarifa Média (R$/kWh)': df['tarifa_media'].mean(),
    'Emissão Total CO₂ (Mt)': df['emissao_co2'].sum() / 1e6,
    'Eficiência Média (%)': df['eficiencia_energetica'].mean(),
    'Consumo Per Capita Médio (kWh)': df['consumo_per_capita'].mean(),
}

print('\n📌 KPIs do Projeto\n' + '='*45)
for k, v in kpis.items():
    if isinstance(v, float):
        print(f'{k}: {v:,.2f}')
    else:
        print(f'{k}: {v}')

### 9. Visualizações

In [ ]:
# 9.1 Evolução Temporal
df_temp = df.groupby('periodo')['consumo_mwh'].sum().reset_index()
fig = px.line(df_temp, x='periodo', y='consumo_mwh',
              title='Evolução Temporal do Consumo de Energia Elétrica',
              labels={'periodo':'Período','consumo_mwh':'Consumo (MWh)'},
              color_discrete_sequence=['#f39c12'], markers=True)
fig.update_layout(hovermode='x unified', plot_bgcolor='#f9f9f9')
fig.show()

In [ ]:
# 9.2 Ranking de estados
df_uf = df.groupby('uf')['consumo_mwh'].sum().sort_values(ascending=True).reset_index()
fig = px.bar(df_uf, x='consumo_mwh', y='uf', orientation='h',
             title='Consumo Total por Estado (2015–2024)',
             labels={'consumo_mwh':'Consumo (MWh)','uf':'Estado'},
             color='consumo_mwh', color_continuous_scale='YlOrRd')
fig.update_layout(showlegend=False, plot_bgcolor='#f9f9f9')
fig.show()

In [ ]:
# 9.3 Comparação setorial
df_setor = df.groupby('setor_consumo')['consumo_mwh'].sum().reset_index()
fig = px.pie(df_setor, names='setor_consumo', values='consumo_mwh',
             title='Distribuição do Consumo por Setor',
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.show()

In [ ]:
# 9.4 Dispersão temperatura x consumo
fig = px.scatter(df, x='temperatura_media', y='consumo_mwh',
                 color='regiao', size='demanda_pico', opacity=0.6,
                 hover_data=['uf','setor_consumo','ano'],
                 title='Relação entre Temperatura Média e Consumo de Energia',
                 labels={'temperatura_media':'Temperatura (°C)','consumo_mwh':'Consumo (MWh)'},
                 trendline='ols')
fig.update_layout(plot_bgcolor='#f9f9f9')
fig.show()

In [ ]:
# 9.5 Heatmap de sazonalidade
df_heat = df.groupby(['ano','mes'])['consumo_mwh'].sum().reset_index()
df_pivot = df_heat.pivot(index='ano', columns='mes', values='consumo_mwh')
df_pivot.columns = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
fig = px.imshow(df_pivot, color_continuous_scale='YlOrRd', text_auto='.2s', aspect='auto',
                title='Heatmap de Consumo Mensal por Ano (Sazonalidade)',
                labels=dict(x='Mês', y='Ano', color='Consumo (MWh)'))
fig.show()

In [ ]:
# 9.6 Comparação regional
df_reg = df.groupby(['ano','regiao'])['consumo_mwh'].sum().reset_index()
fig = px.line(df_reg, x='ano', y='consumo_mwh', color='regiao', markers=True,
              title='Evolução do Consumo por Região (2015–2024)',
              labels={'ano':'Ano','consumo_mwh':'Consumo (MWh)','regiao':'Região'},
              color_discrete_sequence=px.colors.qualitative.Set1)
fig.update_layout(plot_bgcolor='#f9f9f9')
fig.show()

In [ ]:
# 9.7 Evolução da eficiência energética
df_efic = df.groupby(['ano','regiao'])['eficiencia_energetica'].mean().reset_index()
fig = px.line(df_efic, x='ano', y='eficiencia_energetica', color='regiao', markers=True,
              title='Evolução da Eficiência Energética por Região',
              labels={'ano':'Ano','eficiencia_energetica':'Eficiência (%)','regiao':'Região'})
fig.update_layout(plot_bgcolor='#f9f9f9')
fig.show()

In [ ]:
# 9.8 Demanda de pico por nível
df_nivel = df.groupby(['ano','nivel_demanda'])['demanda_pico'].mean().reset_index()
fig = px.bar(df_nivel, x='ano', y='demanda_pico', color='nivel_demanda', barmode='group',
             title='Demanda de Pico Média por Nível e Ano',
             labels={'ano':'Ano','demanda_pico':'Demanda Pico (MW)','nivel_demanda':'Nível'},
             color_discrete_map={'Baixo':'#2ecc71','Médio':'#f39c12','Alto':'#e67e22','Crítico':'#e74c3c'},
             category_orders={'nivel_demanda':['Baixo','Médio','Alto','Crítico']})
fig.show()

### 10. Interpretação dos Resultados

In [ ]:
# Correlação temperatura x consumo por região
corr_regiao = df.groupby('regiao').apply(
    lambda g: g[['temperatura_media','consumo_mwh']].corr().iloc[0,1]
).reset_index()
corr_regiao.columns = ['Região', 'Correlação Temp × Consumo']
print('📊 Correlação entre Temperatura e Consumo por Região:')
print(corr_regiao.to_string(index=False))

In [ ]:
# Crescimento do consumo por região (2015 vs 2024)
cresc = df.groupby(['ano','regiao'])['consumo_mwh'].sum().unstack()
crescimento = ((cresc.loc[2024] - cresc.loc[2015]) / cresc.loc[2015] * 100).reset_index()
crescimento.columns = ['Região', 'Crescimento (%)']
crescimento = crescimento.sort_values('Crescimento (%)', ascending=False)
print('📈 Crescimento do Consumo 2015→2024 por Região:')
print(crescimento.to_string(index=False))

### 11. Conclusão

**Principais achados da análise:**

1. **Crescimento consistente**: O consumo de energia elétrica cresceu ao longo de 2015–2024, com tendência de aceleração nas regiões Norte e Centro-Oeste.

2. **Concentração regional**: O Sudeste mantém a maior demanda absoluta, puxado por SP, RJ e MG.

3. **Sazonalidade**: Meses de verão (Jan, Fev, Dez) apresentam picos de consumo, influenciados pelo calor e maior uso de ar-condicionado.

4. **Temperatura e consumo**: Há correlação positiva entre temperatura e consumo, especialmente nas regiões Norte e Nordeste.

5. **Setor Industrial lidera**: O setor industrial responde pela maior parcela do consumo, seguido do residencial.

6. **Melhora na eficiência**: O índice de eficiência energética apresentou crescimento ao longo do período, indicando políticas públicas efetivas.

7. **Emissões de CO₂**: Associadas principalmente à geração termelétrica acionada em períodos de seca nas hidrelétricas.

**Recomendações:**
- Expandir programas de eficiência energética no setor rural e residencial
- Investir em energias renováveis nas regiões de crescimento acelerado
- Monitorar picos de demanda crítica para evitar blackouts